# 06 CARE Validation: Runtime Governance Checks
**Repo:** local_contexts_geospatial  
**Author:** Lilly Jones, PhD, Daear Consulting, LLC  

## What This Notebook Covers
The CARE Principles establish that data governance is an ongoing
responsibility, not a one-time label assignment. CARE checks should
run at key decision points in any workflow:

- Before using a dataset for a new purpose
- Before exporting a result to a new destination
- Before sharing data with a new collaborator
- Before publishing analysis results

This notebook demonstrates the validation functions in `localcontexts.validation`:

1. `validate_usage()` checks that intended use is compatible with label
2. `validate_label_present()` warn when a dataset lacks a label it should have
3. `validate_export_ready()` comprehensive pre-export check
4. `check_collective_benefit()` CARE-C: document community benefit
5. `check_authority_to_control()` CARE-A: document consent status
6. The "what happens when you violate a label" demonstration

## CARE Principles Quick Reference
| Principle | The question to ask |
|---|---|
| **C** : Collective Benefit | Does this use benefit the originating community? |
| **A** : Authority to Control | Does the community have authority over this use? |
| **R** : Responsibility | Am I accountable to the community, not just my institution? |
| **E** : Ethics | Does this use align with Indigenous values and rights? |

In [20]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json
import warnings

from localcontexts.labels import TKLabel, BCLabel, TKMetadata, BCMetadata
from localcontexts.validation import (
    TKViolationError,
    BCViolationError,
    ProvenanceError,
    MissingLabelWarning,
    validate_usage,
    validate_label_present,
    validate_provenance_intact,
    validate_export_ready,
    check_collective_benefit,
    check_authority_to_control,
)
from localcontexts.propagation import propagate_labels

warnings.simplefilter("always")

print("Imports complete.")

Imports complete.


## `validate_usage()` Label Restriction Enforcement

In [21]:
# Dataset with TK Non-Commercial label
tk = TKMetadata(
    label     = TKLabel.NON_COMMERCIAL,
    community = "Oglala Lakota Nation",
    authority = "Tribal Data Governance Office",
    usage     = "Non-commercial environmental research only",
    contact   = "data@oglalalakota.org",
)
labeled_meta = tk.attach({"name": "pine_ridge_vegetation"})

# Test multiple use types
use_scenarios = [
    ("research",       "Academic environmental research"),
    ("publication",    "Journal article (non-commercial)"),
    ("grant_reporting","Federal grant reporting"),
    ("commercial",     "Commercial wildfire risk product"),
    ("for_profit",     "For-profit consulting report"),
]

print(f"TK Label: {labeled_meta['tk:label']}")
print(f"Community: {labeled_meta['tk:community']}")
print()
print("Use validation results:")

for use, description in use_scenarios:
    try:
        result = validate_usage(labeled_meta, intended_use=use,
                                raise_on_violation=True)
        print(f"  {description:<40} PERMITTED")
    except TKViolationError:
        print(f"  {description:<40} BLOCKED")

TK Label: TK Non-Commercial
Community: Oglala Lakota Nation

Use validation results:
  Academic environmental research          PERMITTED
  Journal article (non-commercial)         PERMITTED
  Federal grant reporting                  PERMITTED
  Commercial wildfire risk product         BLOCKED
  For-profit consulting report             BLOCKED


In [22]:
# Test Community Use Only label (more restrictive)
tk_cuo = TKMetadata(
    label     = TKLabel.COMMUNITY_USE_ONLY,
    community = "Rosebud Sioux Tribe",
    authority = "RST Cultural Preservation Office",
    usage     = "For use by Rosebud Sioux Tribe members and staff only",
)
cuo_meta = tk_cuo.attach({"name": "rosebud_cultural_sites"})

print(f"TK Label: {cuo_meta['tk:label']}")
print()
for use, description in [
    ("community_internal", "Internal Tribal use"),
    ("research",           "External researcher"),
    ("publication",        "Academic publication"),
]:
    try:
        validate_usage(cuo_meta, intended_use=use)
        print(f"  {description} PERMITTED")
    except TKViolationError as e:
        print(f"  {description} BLOCKED")

TK Label: TK Community Use Only

  Internal Tribal use PERMITTED
  External researcher BLOCKED
  Academic publication BLOCKED


## `validate_label_present()` for proactive labeling reminders

In [23]:
# validate_label_present() warns when a dataset should have a label but doesn't
# Use this as a pre-processing check when working with Tribal land data

# Dataset 1: has a label: no warning
present = validate_label_present(
    labeled_meta,
    context="Pine Ridge vegetation survey",
)
print(f"Labeled dataset: has_label={present}  (no warning)")

print()

# Dataset 2: no label: returns a MissingLabelWarning
unlabeled_meta = {
    "name":   "sd_tribal_ndvi_2022",
    "source": "MODIS MOD13Q1",
    "note":   "Covers Pine Ridge and Rosebud",
}

print("Checking unlabeled dataset (expect a warning):")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    present = validate_label_present(
        unlabeled_meta,
        context="Tribal NDVI dataset",
        warn_only=True,
    )
    if w:
        print(f"  Warning: {str(w[0].message)}")

print(f"  has_label={present}")

Labeled dataset: has_label=True  (no warning)

Checking unlabeled dataset (expect a warning):
  has_label=False


## `validate_export_ready()` as a comprehensive pre-export check

In [24]:
# Run all checks before exporting to a public repository

def run_export_check(meta, intended_use, destination, require_contact=False):
    print(f"\nExport check: use='{intended_use}', destination='{destination}'")
    try:
        report = validate_export_ready(
            meta,
            intended_use    = intended_use,
            destination     = destination,
            require_contact = require_contact,
        )
        for check in report["checks"]:
            status = "Passed" if check["passed"] else "Did not pass"
            print(f"  {status} {check['check']}: {check['note']}")
        print(f"  Result: {'ALL PASSED' if report['all_passed'] else 'ISSUES FOUND'}")
    except (TKViolationError, ValueError) as e:
        print(f"  BLOCKED: {e}")


# Scenario 1: Research use to GitHub: should pass
run_export_check(labeled_meta, "research", "GitHub public repository")

# Scenario 2: Commercial use: should fail
run_export_check(labeled_meta, "commercial", "Commercial data product")

# Scenario 3: Research, require contact info
run_export_check(labeled_meta, "research", "Journal supplement", require_contact=True)

# Scenario 4: Unlabeled dataset: should warn
run_export_check(unlabeled_meta, "research", "Conference presentation")


Export check: use='research', destination='GitHub public repository'
  Passed label_present: TK or BC label found
  Passed usage_compatible: Use 'research' is compatible with label
  Result: ALL PASSED

Export check: use='commercial', destination='Commercial data product'
  BLOCKED: Label restriction violated:
TK label 'TK Non-Commercial' prohibits commercial use. Intended use 'commercial' is not permitted. Contact data@oglalalakota.org for permission.

Export check: use='research', destination='Journal supplement'
  Passed label_present: TK or BC label found
  Passed usage_compatible: Use 'research' is compatible with label
  Passed contact_present: Contact: data@oglalalakota.org
  Result: ALL PASSED

Export check: use='research', destination='Conference presentation'
  Did not pass label_present: No label found — consider attaching one
  Result: ISSUES FOUND


C:\Users\gekek\AppData\Local\Temp\ipykernel_21352\3319193935.py:6: MissingLabelWarning: Exporting unlabeled dataset to Conference presentation. If this involves Indigenous data, attach a Local Contexts label.
  report = validate_export_ready(


## CARE Principle Checks

In [25]:
# CARE-C: Collective Benefit
# Document how the analysis benefits the originating community

working_meta = dict(labeled_meta)

print("CARE-C: Collective Benefit")
check_collective_benefit(
    working_meta,
    benefit_description=(
        "Drought trend analysis will be shared with the Oglala Lakota Nation "
        "Natural Resources Department to support land management planning. "
        "Results will be provided in plain-language summary format."
    )
)
print(f"  care:collective_benefit recorded: {'care:collective_benefit' in working_meta}")

CARE-C: Collective Benefit
CARE — Collective Benefit recorded for Oglala Lakota Nation:
  Drought trend analysis will be shared with the Oglala Lakota Nation Natural Resources Department to support land management planning. Results will be provided in plain-language summary format.
  care:collective_benefit recorded: True


In [26]:
# CARE-A: Authority to Control
# Document whether community consent was obtained

print("CARE-A: Authority to Control")
print()

# Case 1: Consent obtained
meta_consented = dict(working_meta)
check_authority_to_control(
    meta_consented,
    consent_obtained    = True,
    consent_description = (
        "Verbal consent from OLN Natural Resources Director obtained "
        "at meeting on 2024-03-15. Written MOU in progress."
    )
)
print(f"  consent_obtained: {meta_consented.get('care:consent_obtained')}")

print()

# Case 2: Sensitive label, consent NOT obtained: should warn
culturally_sensitive_meta = TKMetadata(
    label     = TKLabel.CULTURALLY_SENSITIVE,
    community = "Rosebud Sioux Tribe",
    authority = "RST Cultural Preservation Office",
    usage     = "Restricted access",
).attach({"name": "rsb_cultural_sites"})

print("CARE-A check for Culturally Sensitive label without consent (expect warning):")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    check_authority_to_control(
        culturally_sensitive_meta,
        consent_obtained = False,
    )
    if w:
        print(f"  Warning: {str(w[0].message)[:120]}...")

CARE-A: Authority to Control

  consent_obtained: True

CARE-A check for Culturally Sensitive label without consent (expect warning):


## The Violation: What Happens When You Ignore a Label

In [27]:
# This cell demonstrates what happens when a label restriction is violated
# The TKViolationError is designed to be caught and handled, not to crash pipelines

print("Demonstrating TK label violation handling in a pipeline:")

def process_for_client(meta, intended_use, client_name):
    """Example pipeline function that validates before processing."""
    print(f"\nProcessing request for: {client_name}")
    print(f"Intended use: {intended_use}")

    try:
        validate_usage(meta, intended_use=intended_use, raise_on_violation=True)
        print(f"  Validation passed: proceeding with analysis")
        # ... actual processing would happen here ...
        return {"status": "completed", "client": client_name}

    except TKViolationError as e:
        print(f"  Request blocked by TK label restriction")
        print(f"  Label: {meta.get('tk:label')}")
        print(f"  Community: {meta.get('tk:community')}")
        print(f"  Contact: {meta.get('tk:contact', 'not specified')}")
        print(f"  To proceed: contact the authority above for authorization")
        return {"status": "blocked", "reason": str(e)}


# Legitimate use
process_for_client(labeled_meta, "research", "University lab")

# Attempted commercial use
process_for_client(labeled_meta, "commercial", "Wildfire analytics company")

Demonstrating TK label violation handling in a pipeline:

Processing request for: University lab
Intended use: research
  Validation passed: proceeding with analysis

Processing request for: Wildfire analytics company
Intended use: commercial
  Request blocked by TK label restriction
  Label: TK Non-Commercial
  Community: Oglala Lakota Nation
  Contact: data@oglalalakota.org
  To proceed: contact the authority above for authorization


{'status': 'blocked',
 'reason': "Label restriction violated:\nTK label 'TK Non-Commercial' prohibits commercial use. Intended use 'commercial' is not permitted. Contact data@oglalalakota.org for permission."}

## Summary: Validation Checklist
Add these checks to any workflow that handles labeled Indigenous data:

- At data load: warn if missing label
validate_label_present(meta, context="my dataset")

- Before any analysis: confirm intended use is permitted
validate_usage(meta, intended_use="research")

- After transformations: confirm label wasn't dropped
validate_provenance_intact(derived_meta)

- Before export: run comprehensive check
validate_export_ready(meta, intended_use="research", destination="GitHub")

- Document CARE compliance
check_collective_benefit(meta, benefit_description="...")
check_authority_to_control(meta, consent_obtained=True, consent_description="...")


## Next Notebook
**07 IEEE 2890 Provenance:** The full lifecycle demonstration from data ingest
to final export with a machine-readable provenance chain aligned
to IEEE 2890-2025.